# NBA Game Outcome Prediction
## Notebook 3: K-Nearest Neighbors (KNN) Classifier
**Member 3 Contribution**

This notebook applies a **K-Nearest Neighbors (KNN)** algorithm to predict whether the home team wins an NBA game.

**Dataset:** NBA Games Dataset — https://www.kaggle.com/datasets/nathanlauga/nba-games


## 0. Install Required Packages

In [ ]:
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn']

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('All packages installed successfully!')

## 0.1 Download Dataset

In [ ]:
import os
import subprocess
import sys

if not os.path.exists('games.csv'):
    print('games.csv not found. Downloading dataset...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    path = kagglehub.dataset_download('nathanlauga/nba-games')
    print('Dataset downloaded to:', path)
    # Copy games.csv to current directory
    import shutil
    for root, dirs, files in os.walk(path):
        for file in files:
            if file == 'games.csv':
                shutil.copy2(os.path.join(root, file), 'games.csv')
                print('games.csv copied to current directory!')
                break
else:
    print('games.csv already exists. Skipping download.')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('games.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Visualize distributions of key features
features = [
    'FG_PCT_home', 'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home',
    'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away'
]
target = 'HOME_TEAM_WINS'

df_clean = df[features + [target]].dropna()

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, feat in enumerate(features):
    ax = axes[i // 5][i % 5]
    df_clean[feat].hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(feat, fontsize=8)
plt.suptitle('Feature Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('knn_feature_distributions.png', dpi=150)
plt.show()

print(f'Clean dataset shape: {df_clean.shape}')
print(f'Home Win Rate: {df_clean[target].mean()*100:.2f}%')

## 3. Data Preprocessing

In [ ]:
X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# KNN is distance-based — scaling is critical!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 4. Find Optimal K Value

In [ ]:
# Test k from 1 to 30 and find the best k
k_range = range(1, 31)
train_scores = []
test_scores  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

best_k = k_range[np.argmax(test_scores)]
print(f'Best K: {best_k} with Test Accuracy: {max(test_scores)*100:.2f}%')

# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_range, [s*100 for s in train_scores], label='Train Accuracy', marker='o', markersize=4)
plt.plot(k_range, [s*100 for s in test_scores],  label='Test Accuracy',  marker='s', markersize=4)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Accuracy (%)')
plt.title('KNN — Accuracy vs K Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('knn_k_selection.png', dpi=150)
plt.show()

## 5. Train Final KNN Model

In [ ]:
# Train with best K
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
print(f'KNN model trained with K={best_k}')

## 6. Evaluate Model

In [ ]:
y_pred = knn_best.predict(X_test_scaled)
y_prob = knn_best.predict_proba(X_test_scaled)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 40)
print(f'    KNN (K={best_k}) — TEST RESULTS')
print('=' * 40)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 40)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Away Wins','Home Wins']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Away Wins','Home Wins'],
            yticklabels=['Away Wins','Home Wins'])
plt.title(f'KNN (K={best_k}) — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='green', lw=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'KNN (K={best_k}) — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('knn_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# Cross-validation
cv_scores = cross_val_score(knn_best, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

In [ ]:
# Compare distance metrics
metrics = ['euclidean', 'manhattan', 'chebyshev']
metric_scores = []
for m in metrics:
    knn_m = KNeighborsClassifier(n_neighbors=best_k, metric=m)
    knn_m.fit(X_train_scaled, y_train)
    metric_scores.append(accuracy_score(y_test, knn_m.predict(X_test_scaled)) * 100)

plt.figure(figsize=(6,4))
plt.bar(metrics, metric_scores, color=['#1abc9c','#3498db','#9b59b6'], edgecolor='black')
plt.title(f'KNN (K={best_k}) — Accuracy by Distance Metric')
plt.ylabel('Accuracy (%)')
plt.xlabel('Distance Metric')
plt.ylim(min(metric_scores)-5, 100)
for i, v in enumerate(metric_scores):
    plt.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('knn_metric_comparison.png', dpi=150)
plt.show()

## 7. All-Model Comparison (Summary)

Run this cell after completing all 3 notebooks to compare results manually.

| Algorithm | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|-----------|----------|-----------|--------|----------|---------|
| Random Forest | — | — | — | — | — |
| SVM | — | — | — | — | — |
| KNN | — | — | — | — | — |

*(Fill in values after running all notebooks)*


## 8. Summary

**Key Observations:**
- KNN is a non-parametric, instance-based learning algorithm.
- It classifies a point based on the majority class of its K nearest neighbors.
- Choosing the right K is critical: too small leads to overfitting, too large leads to underfitting.
- Feature scaling is mandatory since KNN relies on distance calculations.
- The elbow method on the K vs Accuracy plot helps identify the optimal K.


## 7. Interactive Prediction Demo (Viva)
Use this section to demonstrate the model's prediction on a single game scenario.

In [ ]:
import numpy as np
import pandas as pd

# 1. Select a random game from the test set for demonstration
random_idx = np.random.randint(0, len(X_test))
sample_game = X_test.iloc[random_idx]
actual_result = y_test.iloc[random_idx]

print('--- Game Scenario Features ---')
print(sample_game)
print('-' * 30)

# 2. Transform the sample for prediction
# Note: scaler.transform expects a 2D array-like input
sample_scaled = scaler.transform([sample_game.values])

# 3. Make prediction
# Automatically find the available model variable
model = None
for var_name in ['best_rf', 'best_svm', 'knn_best', 'best_knn', 'rf_model', 'svm_model', 'knn_model']:
    if var_name in globals():
        model = globals()[var_name]
        break

if model is None:
    print("Error: No trained model found in global scope.")
else:
    prediction = model.predict(sample_scaled)[0]
    proba = model.predict_proba(sample_scaled)[0]

    # 4. Display Results
    print(f'\n[VIVA DEMO RESULTS]')
    if prediction == 1:
        print('>>> PREDICTION: HOME TEAM WINS! 🏀')
        print(f'Confidence Score: {proba[1]*100:.2f}%')
    else:
        print('>>> PREDICTION: AWAY TEAM WINS! 🏆')
        print(f'Confidence Score: {proba[0]*100:.2f}%')

    result_text = "Home Win" if actual_result == 1 else "Away Win"
    status = "✅ CORRECT" if prediction == actual_result else "❌ INCORRECT"
    print(f'Actual Result: {result_text} ({status})')